# Week 10 Lab: move your project into a database

First do `W10_SQL_Basics.ipynb` (plain SQL, no Python). This lab sends the same SQL from Python, using `sqlite3`.

Until now your users lived in `users.txt` and your data in `raw.csv`. Today both move into one SQLite database file,
`DATA/app.db`. The code also grows up: instead of one long file, it becomes a **package** called `app_model`,
where each file has one job.

```
CST1510_CW2/
  fetch_data.py            (Week 8, unchanged)
  load_data.py             copies users.txt and raw.csv into the database
  app_model/
    __init__.py            makes the folder a package (an empty file)
    db.py                  connect to the database
    schema.py              create the tables
    users.py               hashing, register, login, user SQL (from Week 9)
    records.py             your API data: read, search, add, edit, delete
  DATA/raw.csv    DATA/app.db (on your computer only)
```

`auth.py` from Week 9 is replaced by `app_model/users.py`. Its two hash functions are copied across unchanged.
Ideas to revisit: topics `01` to `05` in this week's folder. Topic `05` explains packages and imports, which Step 5 needs.

In [ ]:
# Run this cell first, every time you open the notebook.
import sys
from pathlib import Path

# Your project lives in the CST1510_CW2 folder. This notebook looks for it in the usual places.
# If it cannot find it, paste the full path between the quotes, for example "C:/Users/sam/Documents/CST1510_CW2"
MY_FOLDER = ""


def find_project():
    if MY_FOLDER:
        return Path(MY_FOLDER)
    here = Path.cwd()
    for folder in [here, *here.parents]:
        if folder.name == "CST1510_CW2":
            return folder
    for base in [Path.home(), Path.home() / "Documents", Path.home() / "Desktop", Path.home() / "OneDrive" / "Documents"]:
        if (base / "CST1510_CW2").exists():
            return base / "CST1510_CW2"
    return here


PROJECT = find_project()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project folder:", PROJECT)
if PROJECT.name != "CST1510_CW2":
    print("\nI could not find your CST1510_CW2 folder. Paste its full path into MY_FOLDER above, then run this cell again.")


def show_checks(checks):
    """Print PASS or FAIL for each (passed, description) pair."""
    for passed, description in checks:
        print("PASS" if passed else "FAIL", "-", description)
    print("\nAll checks passed." if all(p for p, _ in checks) else "\nFix the lines marked FAIL, save, and run again.")


show_checks([
    (PROJECT.name == "CST1510_CW2",                f"working in the CST1510_CW2 folder (this is {PROJECT.name})"),
    ((PROJECT / "auth.py").exists(),               "auth.py from Week 9 is here"),
    ((PROJECT / "DATA" / "raw.csv").exists(),      "DATA/raw.csv from Week 8 is here"),
])

## Step 1: Python meets SQLite

`sqlite3` comes with Python. You **connect** to a file, get a **cursor**, `execute` SQL, then `fetchall()` the answer.
`conn.commit()` saves a change. `":memory:"` is a temporary database for practice.

**Try it:**

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE pets (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, kind TEXT, age INTEGER)")
cur.execute("INSERT INTO pets (name, kind, age) VALUES (?, ?, ?)", ("Rex", "dog", 4))
cur.execute("INSERT INTO pets (name, kind, age) VALUES (?, ?, ?)", ("Mia", "cat", 2))
conn.commit()

cur.execute("SELECT * FROM pets WHERE age > ?", (3,))
print(cur.fetchall())

`fetchall()` returns every row (a list of tuples). `fetchone()` returns one row, or `None` when there is none.

**Exercise.** Add a third pet, then write a query that shows only the cats.

## Step 2: Why the `?` matters

The `?` is a **placeholder**. The database treats what you pass in as plain data, never as SQL.
If you build the SQL text with an f-string, a user can type SQL into your form. That is **SQL injection**.

**Try it.** The user types a malicious name:

In [ ]:
typed = "nobody' OR '1'='1"

cur.execute(f"SELECT * FROM pets WHERE name = '{typed}'")          # f-string: DANGEROUS
print("f-string returned:    ", cur.fetchall())

cur.execute("SELECT * FROM pets WHERE name = ?", (typed,))         # placeholder: safe
print("placeholder returned: ", cur.fetchall())

The f-string version returned everybody. The placeholder version returned nothing.

**Rule: values always go in with `?`.** One more rule from the same family: an `UPDATE` or `DELETE` without a `WHERE`
changes **every** row.

## Step 3: Functions that take `conn`

Each job becomes a small function that receives the connection. This is the shape of all the code you build today.

**Try it:**

In [ ]:
def create_pets_table(conn):
    cur = conn.cursor()
    cur.execute("CREATE TABLE IF NOT EXISTS pets2 (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL UNIQUE, age INTEGER)")
    conn.commit()


def add_pet(conn, name, age):
    cur = conn.cursor()
    cur.execute("INSERT INTO pets2 (name, age) VALUES (?, ?)", (name, age))
    conn.commit()


def get_pet(conn, name):
    cur = conn.cursor()
    cur.execute("SELECT * FROM pets2 WHERE name = ?", (name,))
    return cur.fetchone()


create_pets_table(conn)
add_pet(conn, "Rex", 4)
print(get_pet(conn, "Rex"))
print(get_pet(conn, "Nobody"))

try:
    add_pet(conn, "Rex", 9)                      # the UNIQUE rule is broken
except sqlite3.IntegrityError as error:
    print("Caught:", error)
conn.close()

`UNIQUE` made the database refuse a second `Rex`. Python raises `sqlite3.IntegrityError`, and your code can catch it.

## Step 4: pandas and SQL

pandas moves whole tables in and out of a database in one line each:

| Job | Code |
|---|---|
| CSV file to a table | `df = pd.read_csv("file.csv")` then `df.to_sql("name", conn, if_exists="replace", index=False)` |
| table to a DataFrame | `pd.read_sql("SELECT * FROM name", conn)` |

`pd.read_sql` can also take values with `?`, exactly like `execute`: `pd.read_sql("SELECT * FROM records WHERE mag > ?", conn, params=(4,))`.

**Try it.** This loads your own `raw.csv` into a practice database and reads it back:

In [ ]:
import pandas as pd

practice = sqlite3.connect(":memory:")
df = pd.read_csv(PROJECT / "DATA" / "raw.csv")
df.to_sql("records", practice, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM records LIMIT 5", practice)

**Exercise.** Change the query to count the rows, then to show only rows where one of your number columns is above its average.
(Look at the SQL Basics notebook for `COUNT`, `WHERE` and `AVG`.)

## Step 5: Build the `app_model` package

A **package** is a folder with an (empty) `__init__.py` file. Python can then import from it:
`from app_model.db import connect_database`. (Topic `05` covers this. Remember: Python looks for `app_model` in the folder of the script you run, so keep `app_model` next to `load_data.py`.)

**Build.** Create the folder `app_model` inside `CST1510_CW2`, and an **empty** file `app_model/__init__.py` inside it.
Then create the four files below, one at a time.

### `app_model/db.py`: connect

```python
"""
db.py - Week 10: how the rest of the project connects to the database.
"""

import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent.parent / "DATA" / "app.db"


def connect_database(path=DB_PATH):
    """Open the database file (it is created if it does not exist) and return the connection."""
    path.parent.mkdir(exist_ok=True)
    return sqlite3.connect(path)
```

### `app_model/schema.py`: the users table

`UNIQUE COLLATE NOCASE` means no two usernames can match, even if the capitals differ. `role` is `'user'` unless you say otherwise.

```python
"""
schema.py - Week 10: create the tables.
"""


def create_user_table(conn):
    """Create the users table if it is not there yet."""
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            username      TEXT NOT NULL UNIQUE COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            role          TEXT DEFAULT 'user'
        )
    """)
    conn.commit()
```

### `app_model/users.py`: everything about users

The top is Week 9's two hash functions, unchanged. Then the four SQL jobs (INSERT, SELECT, UPDATE, DELETE). At the bottom,
`register_user` and `login_user` are the Week 9 functions again, now using the database. They still return `(True or False, message)`.

```python
"""
users.py - Week 10: everything about users, now stored in the database.

generate_hash and is_valid_hash are the Week 9 functions, unchanged.
register_user and login_user keep the same shape: they return (True or False, message).
"""

import sqlite3

import bcrypt


# ---- passwords (Week 9) -----------------------------------------------------

def generate_hash(password):
    """Turn a password into a bcrypt hash (returned as text)."""
    byte_password = password.encode("utf-8")
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(byte_password, salt)
    return hashed.decode("utf-8")


def is_valid_hash(password, hashed):
    """True if the password matches the stored hash."""
    return bcrypt.checkpw(password.encode("utf-8"), hashed.encode("utf-8"))


# ---- SQL: create, read, update, delete -------------------------------------

def add_user(conn, username, password_hash, role="user"):
    """INSERT one user. Raises sqlite3.IntegrityError if the username is taken."""
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO users (username, password_hash, role) VALUES (?, ?, ?)",
        (username, password_hash, role),
    )
    conn.commit()


def get_user(conn, username):
    """SELECT one user: (id, username, password_hash, role), or None."""
    cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE username = ?", (username,))
    return cur.fetchone()


def get_all_users(conn):
    """SELECT every user."""
    cur = conn.cursor()
    cur.execute("SELECT * FROM users")
    return cur.fetchall()


def update_role(conn, username, new_role):
    """UPDATE one user's role."""
    cur = conn.cursor()
    cur.execute("UPDATE users SET role = ? WHERE username = ?", (new_role, username))
    conn.commit()


def delete_user(conn, username):
    """DELETE one user. (Without the WHERE it would delete everybody!)"""
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE username = ?", (username,))
    conn.commit()


# ---- register and log in (Week 9, now using the database) --------------------

def register_user(conn, username, password):
    """Add a new user. Returns (True or False, a message)."""
    username = username.strip()
    if username == "":
        return False, "Username cannot be empty."
    if len(password) < 8:
        return False, "Password must be at least 8 characters."
    try:
        add_user(conn, username, generate_hash(password))
    except sqlite3.IntegrityError:          # the UNIQUE rule on username
        return False, "That username is already taken."
    return True, "Registered. You can log in now."


def login_user(conn, username, password):
    """Check a username and password. Returns (True or False, a message)."""
    user = get_user(conn, username.strip())
    if user is not None and is_valid_hash(password, user[2]):
        return True, "Welcome back!"
    return False, "Wrong username or password."      # same message for both problems


def migrate_users_file(conn, path):
    """Copy the users from the Week 9 users.txt into the database. Returns how many were added."""
    if not path.exists():
        return 0
    added = 0
    with open(path, "r") as file:
        for line in file:
            name, password_hash = line.strip().split(",", 1)
            if get_user(conn, name) is None:
                add_user(conn, name, password_hash)
                added += 1
    return added
```

### `app_model/records.py`: your API data

`load_csv` is Step 4. The other functions read, search, add, edit and delete rows. Each row is found by its `row_id`.
`_check_column` exists because a `?` can protect a **value** but not a **column name**, so names are checked against the real list.

Three small things you have not met yet in `records.py`:

- `f"... FROM {TABLE}"` is fine here because `TABLE` is a constant **you** wrote. An f-string is only dangerous when a *user* can type into it; the table name never comes from a user. Values still use `?`.
- `cur.description` lists the columns of the last query; the first item of each is the column name. `LIMIT 0` asks for no rows at all, so `get_columns` gets the names and nothing else.
- `SELECT rowid AS row_id, *` adds SQLite's own hidden row number, so each row can be found again to edit or delete it.

```python
"""
records.py - Week 10: your API data, stored in the 'records' table.

The table is created from raw.csv, so its columns are whatever your data has.
Each row can be found by its row_id (SQLite's built-in rowid).
"""

import pandas as pd

TABLE = "records"


def load_csv(conn, csv_path):
    """Copy a CSV file into the records table (replacing what was there). Returns the number of rows."""
    df = pd.read_csv(csv_path)
    df.to_sql(TABLE, conn, if_exists="replace", index=False)
    return len(df)


def get_columns(conn):
    """The column names of the records table."""
    cur = conn.cursor()
    cur.execute(f"SELECT * FROM {TABLE} LIMIT 0")
    return [column[0] for column in cur.description]


def get_all_records(conn):
    """Every row, as a table, with a row_id column at the front."""
    return pd.read_sql(f"SELECT rowid AS row_id, * FROM {TABLE}", conn)


def count_records(conn):
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {TABLE}")
    return cur.fetchone()[0]


def _check_column(conn, column):
    """A ? can protect a value but not a column NAME, so names are checked against the real list."""
    if column not in get_columns(conn):
        raise ValueError(f"Unknown column: {column}")


def search_records(conn, column, text):
    """Rows where a column contains some text."""
    _check_column(conn, column)
    sql = f"SELECT rowid AS row_id, * FROM {TABLE} WHERE {column} LIKE ?"
    return pd.read_sql(sql, conn, params=(f"%{text}%",))


def add_record(conn, values):
    """INSERT one row. 'values' is a dictionary {column: value}."""
    for column in values:
        _check_column(conn, column)
    names = ", ".join(values)
    marks = ", ".join("?" for _ in values)
    cur = conn.cursor()
    cur.execute(f"INSERT INTO {TABLE} ({names}) VALUES ({marks})", tuple(values.values()))
    conn.commit()


def update_record(conn, row_id, column, value):
    """UPDATE one value in one row. Returns how many rows changed (0 if the row_id does not exist)."""
    _check_column(conn, column)
    cur = conn.cursor()
    cur.execute(f"UPDATE {TABLE} SET {column} = ? WHERE rowid = ?", (value, row_id))
    conn.commit()
    return cur.rowcount


def delete_record(conn, row_id):
    """DELETE one row. Returns how many rows were deleted."""
    cur = conn.cursor()
    cur.execute(f"DELETE FROM {TABLE} WHERE rowid = ?", (row_id,))
    conn.commit()
    return cur.rowcount
```

## Step 6: Fill the database

**Build.** Create `load_data.py` in the `CST1510_CW2` folder (not inside `app_model`):

```python
"""
load_data.py - Week 10: fill the database. Safe to run again.

Run it:  python load_data.py
"""

from pathlib import Path

from app_model.db import connect_database
from app_model.records import count_records, load_csv
from app_model.schema import create_user_table
from app_model.users import migrate_users_file

DATA_FOLDER = Path(__file__).parent / "DATA"


if __name__ == "__main__":
    conn = connect_database()
    create_user_table(conn)

    csv_file = DATA_FOLDER / "raw.csv"
    if csv_file.exists():
        load_csv(conn, csv_file)
        print(f"Loaded {count_records(conn)} rows into the records table.")
    else:
        print("DATA/raw.csv not found - run fetch_data.py first.")

    moved = migrate_users_file(conn, DATA_FOLDER / "users.txt")
    print(f"Moved {moved} user(s) from users.txt into the users table.")
    conn.close()
```

Run it from the terminal inside `CST1510_CW2`. It is safe to run again:

```
python load_data.py
```

You should see something like `Loaded 500 rows into the records table.` and `Moved 1 user(s) from users.txt ...`
(the number of users is however many you registered in Week 9).

## Step 7: Check everything

This test uses a temporary database, so it will not change yours.

In [ ]:
import importlib, tempfile
import app_model.db, app_model.schema, app_model.users, app_model.records
for module in (app_model.db, app_model.schema, app_model.users, app_model.records):
    importlib.reload(module)
from app_model.db import connect_database
from app_model.schema import create_user_table
from app_model.users import register_user, login_user, get_user
from app_model.records import load_csv, count_records, get_columns, get_all_records, search_records, add_record, update_record, delete_record

test_db = connect_database(Path(tempfile.mkdtemp()) / "test.db")
create_user_table(test_db)

rows = load_csv(test_db, PROJECT / "DATA" / "raw.csv")
columns = get_columns(test_db)
text_col = get_all_records(test_db).select_dtypes(exclude="number").columns[0]
sample = str(get_all_records(test_db)[text_col].dropna().iloc[0])[:3]
number_col = [c for c in get_all_records(test_db).select_dtypes("number").columns if c != "row_id"][0]

add_record(test_db, {text_col: "TEST ROW"})
new_id = int(get_all_records(test_db).iloc[-1]["row_id"])
update_record(test_db, new_id, number_col, 123)
changed = get_all_records(test_db).query("row_id == @new_id").iloc[0][number_col]
delete_record(test_db, new_id)

try:
    search_records(test_db, "not_a_column; DROP TABLE users", "x")
    blocked = False
except ValueError:
    blocked = True

show_checks([
    (register_user(test_db, "anna", "short")[0] is False,           "a short password is refused"),
    (register_user(test_db, "anna", "longenough1")[0] is True,      "register works"),
    (register_user(test_db, "Anna", "longenough1")[0] is False,     "a duplicate username is refused"),
    (login_user(test_db, "anna", "longenough1")[0] is True,         "the right password is accepted"),
    (login_user(test_db, "anna", "wrong-password")[0] is False,     "the wrong password is refused"),
    (get_user(test_db, "anna")[2] != "longenough1",                 "the stored value is a hash, not the password"),
    (rows == count_records(test_db) and rows >= 50, f"{rows} rows loaded from raw.csv"),
    (len(search_records(test_db, text_col, sample)) >= 1,           f"search found rows with '{sample}' in {text_col}"),
    (changed == 123,                                                f"update changed {number_col}"),
    (count_records(test_db) == rows,                                "delete put the row count back"),
    (blocked,                                                       "a made-up column name is refused"),
])

Now look at your real database:

In [ ]:
conn = connect_database()
print("Users:", pd.read_sql("SELECT id, username, role FROM users", conn))
print("Rows in records:", count_records(conn))
conn.close()

**Optional.** Install the *SQLite Viewer* (or *SQLite Explorer*) extension in VS Code and click `DATA/app.db` to browse the tables.

## Step 8: Keep the database private

`app.db` holds password hashes, so it stays on your computer (it is in your `.gitignore`). Check:

In [ ]:
import subprocess
result = subprocess.run(["git", "check-ignore", "-v", "DATA/app.db"], cwd=PROJECT, capture_output=True, text=True)
print(result.stdout.strip() or "NOT ignored: add  DATA/app.db  to your .gitignore")

Your users are now in the database, so the Week 9 files are finished with. Delete `auth.py` and `DATA/users.txt` from your folder
(Git remembers `auth.py` in your Week 9 commit). Do this only after `load_data.py` said it moved your users.

## Summary

```
CST1510_CW2/
  fetch_data.py   load_data.py
  app_model/      __init__.py  db.py  schema.py  users.py  records.py
  DATA/           raw.csv   app.db (on your computer only)
  .gitignore
```

**Commit your work.** In the VS Code terminal, inside `CST1510_CW2`:

```
git add .
git commit -m "Week 10: SQLite database and app_model package"
git push
```

Then open `CST1510_CW2` on GitHub. You should see the `app_model` folder but **not** `app.db`.

## Common errors

| You see | Why | Fix |
|---|---|---|
| `sqlite3.OperationalError: no such table: users` | the table was not created | run `python load_data.py`, or call `create_user_table(conn)` |
| `sqlite3.IntegrityError: UNIQUE constraint failed` | that username exists | that is the rule working: `register_user` catches it |
| `sqlite3.ProgrammingError: Incorrect number of bindings` | the number of `?` and values differ | count them; the values must be a tuple like `(a, b)` |
| `sqlite3.OperationalError: database is locked` | another program has the file open | close the SQLite viewer or restart the kernel |
| `ModuleNotFoundError: No module named 'app_model'` | wrong folder, or no `__init__.py` | run from inside `CST1510_CW2`; check `app_model/__init__.py` exists |
| `ValueError: Unknown column` | the name is not in your table | print `get_columns(conn)` and copy the name |

Still stuck? Read the **last line** of the error message, then compare with `DEMO/project/`.